# 17 · Geometry buffer, checked against theory

## What this checks

**Geometry buffer** draws the area within a stated distance of a point. This check covers release 0.7.0. At
release 0.1.0, when it was first checked, the widget was called Area around a point. A true circle cannot be drawn with straight lines, so the widget draws a
polygon with many corners instead. The check asks whether it draws the circle it claims to, and how close that
polygon is to a real circle.

The edge of a buffer decides what counts as inside. So its size matters.

## How it is checked

**This is the first widget in this lab that cannot agree exactly with a second program**, and this check says so.
Fieldwork buffers with Turf, a JavaScript geometry library. Turf draws the circle on a flat map centred on the point,
where distances from the centre are true. No other program builds the circle in that exact way, so there is nothing
equivalent to compare it with. **Buffer study area**, now retired, used the same method at a coarser setting, and has
its own check, 18.

What *can* be checked exactly is the shape of the approximation. A circle drawn with a number of steps in each
quarter is a polygon with 4 × steps corners, sitting just inside the circle. The share of the circle's area that such
a polygon covers has an exact formula, where n is the number of corners:

$$\frac{n}{2\pi}\sin\frac{2\pi}{n}$$

That is 0.99359 at 32 corners and 0.99960 at 128. Those are exactly the figures Fieldwork's own comments record from
measurement. So the share the widget reports can be checked against **theory**, not against another program.

We measure distances on the ground with **pyproj**, the Python link to PROJ, the projection library most GIS software
uses. No code is shared with Fieldwork.

The data are one point in Soho, from the John Snow example, buffered by 500 m with 32 steps in each quarter.

## What should happen

1. **The corners.** Exactly 128 corners: 4 × 32.
2. **The share of the circle.** The share the widget reports is within 0.00005 of the formula. That is the four
   decimal places the widget reports.
3. **The radius.** Every corner is within 1 millimetre of 500 m from the centre. It is measured on the sphere Turf
   uses, with a radius of 6,371,008.8 m.

These criteria were fixed before the check was run. The third was changed once, for the reason given below.

In [ ]:
# Only what this notebook imports. Requesting rasterio alongside pyproj kills this kernel
# outright; see 00-kernel-check for the measured map.
NEEDED = ["pyproj", "shapely", "numpy"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested from the Pyodide distribution:", ", ".join(NEEDED))
except ImportError:
    print("Not a Pyodide kernel; using the ambient environment.")
except Exception as error:
    print("loadPackage failed:", type(error).__name__, error)
    try:
        import piplite
        await piplite.install(NEEDED)
    except Exception as fallback:
        print("piplite also failed:", type(fallback).__name__, fallback)

In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_17_buffer_point import run

report = run(Path.cwd() / "fixtures" / "buffer-point-soho-001.json")
print("criterion:", report["criterion"])
print()
print("agrees:", report["agrees"])
print()
print(f"  steps per quadrant      {report['stepsPerQuadrant']}")
print(f"  vertices                {report['vertexCount']} (expected {report['expectedVertexCount']})")
print(f"  circle-area fraction    {report['reportedCircleAreaFraction']} "
      f"against closed form {report['closedFormFraction']:.5f}")
print(f"  worst radius error      {report['worstVertexRadiusErrorMetres']:.3e} m on the sphere")
print()
print(f"  on the sphere  {report['sphericalRadiusRangeMetres'][0]:.4f} to "
      f"{report['sphericalRadiusRangeMetres'][1]:.4f} m")
print(f"  on WGS84       {report['ellipsoidalRadiusRangeMetres'][0]:.4f} to "
      f"{report['ellipsoidalRadiusRangeMetres'][1]:.4f} m")

## The result on record

First recorded on 11 October 2026, against release 0.1.0. Rerun on 26 November 2026, against release 0.7.0:
**Agrees**.

- **The corners.** There are exactly 128.
- **The share of the circle.** The widget reports 0.9996. The formula gives 0.99960, a difference of about
  1.5 × 10⁻⁶.
- **The radius.** On Turf's sphere, every corner is within about 2.4 × 10⁻⁹ m of 500 m.

Running the cell above repeats the check on the same data.

## The criterion that was wrong, and what it found

**This check failed on its first run**, and the fault was in the criterion, not the widget. It asked that every
corner be within 1 m of 500 m, measured on the WGS84 ellipsoid. The ellipsoid is the slightly flattened shape GPS
uses for the Earth. The worst corner was 1.589 m out.

We looked into it, instead of widening the tolerance. That gave the most useful fact in this notebook. Turf buffers on
a **sphere with a radius of 6,371,008.8 m**, and does so exactly.

In [ ]:
import pyproj

fixture = json.loads((Path.cwd() / "fixtures" / "buffer-point-soho-001.json").read_text())
centre = fixture["input"]["point"]["coordinates"]
ring = fixture["output"]["boundary"]["coordinates"][0][:-1]
radius = fixture["declaredParameters"]["radiusMetres"]

sphere = pyproj.Geod(a=6371008.8, b=6371008.8)
wgs84 = pyproj.Geod(ellps="WGS84")

for label, model in [("sphere 6371008.8 m", sphere), ("WGS84 ellipsoid   ", wgs84)]:
    distances = [model.inv(centre[0], centre[1], x, y)[2] for x, y in ring]
    print(f"{label}  min {min(distances):9.4f}  max {max(distances):9.4f}  "
          f"spread {max(distances) - min(distances):7.4f} m")

print()
print(f"The widget says {radius} m. On the sphere it draws {radius} m, to four decimals, in every")
print("direction. On the ellipsoid the same ring is up to 1.59 m further out, and by a different")
print("amount in each direction -- which is why the spread is not zero there.")
print()
print(f"As a fraction of the radius that is {1.589 / radius:.2%}. On a 5 km scope it would be")
print(f"about {1.589 / radius * 5000:.0f} m.")

### Why this matters more than the tolerance did

A practitioner who reads "everything within 500 m of this point" will take it to mean 500 m on the ground. It is
500 m on a sphere. On the ground it is **500.28 to 501.59 m**, depending on direction.

Nothing is wrong with the widget. A buffer on a sphere is a reasonable method, and the error is a third of a percent.
That is about 16 m on a 5 km scope. But this was not written down anywhere before this check was attempted. It was
found only because a criterion fixed in advance failed, and was looked into, not relaxed.

That is the case for fixing criteria before running. A tolerance chosen after seeing the answer would have been 2 m.
It would have passed, and found nothing.

## What this does not show

- That a 500 m buffer is 500 m on the ground. It is 500 m on a sphere with a radius of 6,371,008.8 m. On the WGS84
  ellipsoid at this latitude, it is 500.28 to 501.59 m, up to 1.59 m more, depending on direction. That is how the
  method works. It is reported here, not corrected.
- That the radius is the right radius. A buffered scope is a decision, not a surveyed boundary. Its edge decides
  what is inside, and its radius is an assumption of the question.